# 🏡 NYC Airbnb Dataset: Data Cleaning, Standardization & Preparation for Power BI

## Introduction & Project Scope

This project focuses on the end-to-end data cleaning, validation, and standardization of the New York City Airbnb dataset to prepare it for high-integrity Business Intelligence (Power BI) reporting. Real-world marketplace data often suffers from inconsistencies, missing spatial metadata, typographical errors, and unverified records. Rather than applying naive row deletions or synthetic imputations—which introduce analytical bias—this pipeline enforces a strict, data-integrity-first approach. By combining domain-specific logic, hierarchical string standardization, spatial proximity analysis (k-NN), and file format optimizations (Parquet), this preparation layer guarantees that downstream analytical models and KPIs reflect factual operational realities.

## <span style=" color:red"> Import Libraries, Loading the Dataset and Initial Exploration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import missingno as msno
import plotly.express as px
import folium
from folium import plugins

pd.set_option("display.max_columns", None) 
pd.set_option('display.max_rows', None)

import warnings
warnings.filterwarnings("ignore") 

In [2]:
df0 = pd.read_csv("Airbnb_Open_Data.csv")
df = df0.copy()

In [3]:
df.head(2)

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,country code,instant_bookable,cancellation_policy,room type,Construction year,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,US,False,strict,Private room,2020.0,$966,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,US,False,moderate,Entire home/apt,2007.0,$142,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN


In [4]:
df.info()

# The dataset consists of 102,599 rows and 26 columns with data types distributed as float64 (9), int64 (2), and object (15). 
# Initial checks indicate that all columns contain null values, with the exception of id, host_id, and room_type. 
# Notably, the license column is almost entirely missing, with non-null values present in only 2 records.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  cancellation_pol

In [5]:
df.shape

(102599, 26)

In [6]:
df.columns # Let's inspect the column names.

Index(['id', 'NAME', 'host id', 'host_identity_verified', 'host name',
       'neighbourhood group', 'neighbourhood', 'lat', 'long', 'country',
       'country code', 'instant_bookable', 'cancellation_policy', 'room type',
       'Construction year', 'price', 'service fee', 'minimum nights',
       'number of reviews', 'last review', 'reviews per month',
       'review rate number', 'calculated host listings count',
       'availability 365', 'house_rules', 'license'],
      dtype='object')

In [7]:
# and standardize the column names.

column_names = {
    'NAME': 'name',
    'host id': 'host_id',
    'host_identity_verified': 'host_identity_verified',
    'host name': 'host_name',
    'neighbourhood group': 'neighbourhood_group',
    'room type': 'room_type',
    'Construction year': 'construction_year',
    'service fee': 'service_fee',
    'minimum nights': 'minimum_nights',
    'number of reviews': 'number_of_reviews',
    'last review': 'last_review',
    'reviews per month': 'reviews_per_month',
    'review rate number': 'review_rate_number',
    'calculated host listings count': 'calculated_host_listings_count',
    'availability 365': 'availability_365',
    'country code': 'country_code',
    'instant_bookable': 'instant_bookable',
    'cancellation_policy': 'cancellation_policy',
    'house_rules': 'house_rules'
}

df.rename(columns=column_names, inplace=True)

In [8]:
df.columns

Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country',
       'country_code', 'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules', 'license'],
      dtype='object')

In [9]:
df.dtypes #Checking Data Types

id                                  int64
name                               object
host_id                             int64
host_identity_verified             object
host_name                          object
neighbourhood_group                object
neighbourhood                      object
lat                               float64
long                              float64
country                            object
country_code                       object
instant_bookable                   object
cancellation_policy                object
room_type                          object
construction_year                 float64
price                              object
service_fee                        object
minimum_nights                    float64
number_of_reviews                 float64
last_review                        object
reviews_per_month                 float64
review_rate_number                float64
calculated_host_listings_count    float64
availability_365                  

In [10]:
df["construction_year"] = df["construction_year"].astype("Int64")

# Converted float/numeric columns with missing values to Pandas' nullable integer format (Int64). 
# Standard int64 fails when NaN values are present, 
# whereas Int64 supports missing data while maintaining integer integrity for Power BI export.

In [11]:
# Stripped currency symbols ($), commas, and whitespace from price and service_fee string columns, 
# then converted them to numeric values using pd.to_numeric() with errors='coerce' to handle unparseable entries safely.

df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

df["service_fee"] = (
    df["service_fee"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["service_fee"] = pd.to_numeric(df["service_fee"], errors="coerce")

In [12]:
df["minimum_nights"].dropna().loc[
    df["minimum_nights"].dropna() % 1 != 0
]

# Verified that the floating-point minimum_nights values contain no fractional components other than .0. 
# This ensures that converting minimum_nights from float64 to Int64 proceeds without any data loss.

Series([], Name: minimum_nights, dtype: float64)

In [13]:
df["minimum_nights"] = df["minimum_nights"].astype("Int64")

In [14]:
# Replicated Precision Check;

df["number_of_reviews"].dropna().loc[
    df["number_of_reviews"].dropna() % 1 != 0
]

Series([], Name: number_of_reviews, dtype: float64)

In [15]:
df["number_of_reviews"] = df["number_of_reviews"].astype("Int64")

In [16]:
df["last_review"] = pd.to_datetime(df["last_review"])

# DateTime Conversion: Converted the column to datetime64 format to enable time-series analysis and proper date filtering in Power BI.

In [17]:
# Replicated Precision Check;

df["review_rate_number"].dropna().loc[
    df["review_rate_number"].dropna() % 1 != 0
]

Series([], Name: review_rate_number, dtype: float64)

In [18]:
df["review_rate_number"] = df["review_rate_number"].astype("Int64")

In [19]:
# Replicated Precision Check;

df["calculated_host_listings_count"].dropna().loc[
    df["calculated_host_listings_count"].dropna() % 1 != 0
]

Series([], Name: calculated_host_listings_count, dtype: float64)

In [20]:
df["calculated_host_listings_count"] = df["calculated_host_listings_count"].astype("Int64")

In [21]:
# Replicated Precision Check;

df["availability_365"].dropna().loc[
    df["availability_365"].dropna() % 1 != 0
]

Series([], Name: availability_365, dtype: float64)

In [22]:
df["availability_365"] = df["availability_365"].astype("Int64")

In [23]:
df["instant_bookable"] = (
    df["instant_bookable"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"TRUE": True, "FALSE": False})
)

df["instant_bookable"] = df["instant_bookable"].astype("boolean")

# Boolean Standardization & Null Preservation: 
# Standardized text strings to uppercase (TRUE/FALSE) and mapped them to boolean values. 
# Using Pandas' nullable boolean dtype ensures that original missing (NaN) values remain intact without forcing them to False.

In [24]:
df.dtypes

id                                         int64
name                                      object
host_id                                    int64
host_identity_verified                    object
host_name                                 object
neighbourhood_group                       object
neighbourhood                             object
lat                                      float64
long                                     float64
country                                   object
country_code                              object
instant_bookable                         boolean
cancellation_policy                       object
room_type                                 object
construction_year                          Int64
price                                    float64
service_fee                              float64
minimum_nights                             Int64
number_of_reviews                          Int64
last_review                       datetime64[ns]
reviews_per_month   

In [25]:
df.head(1)

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,US,False,strict,Private room,2020,966.0,193.0,10,9,2021-10-19,0.21,4,6,286,Clean up and treat the home the way you'd like...,NaN


## <span style=" color:red"> General Inspection of Numeric Features 

In [26]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
id,102599.0,29146234.52213,1001254.0,15085814.5,29136603.0,43201198.0,57367417.0,16257505.607309
host_id,102599.0,49254111474.328667,123600518.0,24583328475.0,49117739352.0,73996495817.0,98763129024.0,28538996644.374817
lat,102591.0,40.728094,40.49979,40.68874,40.72229,40.76276,40.91697,0.055857
long,102591.0,-73.949644,-74.24984,-73.98258,-73.95444,-73.93235,-73.70522,0.049521
construction_year,102385.0,2012.487464,2003.0,2007.0,2012.0,2017.0,2022.0,5.765556
price,102352.0,625.293536,50.0,340.0,624.0,913.0,1200.0,331.671614
service_fee,102326.0,125.026924,10.0,68.0,125.0,183.0,240.0,66.325739
minimum_nights,102190.0,8.135845,-1223.0,2.0,3.0,5.0,5645.0,30.553781
number_of_reviews,102416.0,27.483743,0.0,1.0,7.0,30.0,1024.0,49.508954
last_review,86706,2019-06-12 03:40:52.065601024,2012-07-11 00:00:00,2018-10-28 00:00:00,2019-06-14 00:00:00,2019-07-05 00:00:00,2058-06-16 00:00:00,NaN


**Upon inspection**, we observe **extreme anomalies in availability_365 and minimum_nights**, specifically **negative values** and **unrealistic high numbers:**

**1. Outlier Handling for minimum_nights**: Upon inspection, we identified negative values and unrealistic high numbers in the minimum_nights feature. Logically, a minimum stay cannot be less than 1 night, and per New York City short-term rental regulations, it cannot exceed 365 nights.

In [27]:
# Checking the number of records where minimum_nights exceeds 365 nights;

over_365_count = (df["minimum_nights"] > 365).sum()
print(f"Number of rows where minimum_nights exceeds 365 nights: {over_365_count}")

# Displaying these outlier records;

df[df["minimum_nights"] > 365][["id", "name", "minimum_nights"]]

Number of rows where minimum_nights exceeds 365 nights: 35


,id,name,minimum_nights
167,1093570,Convenient cozy cheap apt Manhattan,371
186,1104064,SAFE AND BEAUTIFUL ACCOMODATION,366
263,1146591,"Private, Large & Sunny Top Floor Apt w/W&D",399
299,1166474,Bright Beautiful Brooklyn,452
350,1194641,LARGE 1BR (CONV 2BR) CROWN HEIGHTS,3455
473,1262574,Spacious Quiet rm - 20mins to Midtown,398
1306,1722640,800sqft apartment with huge terrace,370
2855,2578153,NaN,1000
5768,4187002,Prime W. Village location 1 bdrm,1250
7356,5064055,Beautiful Fully Furnished 1 bed/bth,500


In [28]:
# Count and proportion of rows where minimum_nights > 365;

over_365_count = (df["minimum_nights"] > 365).sum()
over_365_ratio = (over_365_count / len(df)) * 100

print(f"365 geceden fazla olan satır sayısı: {over_365_count}")
print(f"Toplam veriye oranı: %{over_365_ratio:.4f}")

365 geceden fazla olan satır sayısı: 35
Toplam veriye oranı: %0.0341


In [29]:
# Displaying rows with negative minimum_nights values; 

df[df["minimum_nights"] < 0][
    ["id", "name", "neighbourhood", "room_type", "minimum_nights"]
]

,id,name,neighbourhood,room_type,minimum_nights
176,1098541,BROOKLYN VICTORIAN STYLE SUITE.....,Fort Greene,Private room,-10
352,1195746,"Beautiful, Bright’s, Warm & Spacious 1.5BR Apt",Crown Heights,Entire home/apt,-5
398,1221151,SUPER BIG AND COZY PRIVATE BEDROOM,Kensington,Private room,-1
421,1233854,Charming Nolita Apartment!!,Nolita,Entire home/apt,-10
441,1244900,Cozy apartment in a brownstone,Harlem,Entire home/apt,-12
478,1265335,Charming upper west side apartment,Upper West Side,Entire home/apt,-2
525,1291294,Chateau Style Brooklyn Loft for Singles or Cou...,Bedford-Stuyvesant,Entire home/apt,-3
42446,24444262,"Cozy room in bright, spacious apartment",Hunts Point,Private room,-1223
42500,24474086,2bd BOUTIQUE Apartament in the heart of MANHA...,Hell's Kitchen,Entire home/apt,-365
42538,24495073,Newly Renovated Garden Apartment,Bedford-Stuyvesant,Entire home/apt,-200


In [30]:
df[df["minimum_nights"] < 0] 

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license
176,1098541,BROOKLYN VICTORIAN STYLE SUITE.....,83631499592,unconfirmed,Frederick,Brooklyn,Fort Greene,40.69098,-73.97113,United States,US,False,flexible,Private room,2008,1155.0,231.0,-10,213,2019-06-24,2.00,5,2,19,There is no smoking on the property. No outsid...,NaN
352,1195746,"Beautiful, Bright’s, Warm & Spacious 1.5BR Apt",2227683633,unconfirmed,Grant,Brooklyn,Crown Heights,40.67174,-73.95663,NaN,NaN,<NA>,NaN,Entire home/apt,2009,73.0,15.0,-5,104,2019-06-21,1.04,5,1,31,We are a second-floor apartment so please do n...,NaN
398,1221151,SUPER BIG AND COZY PRIVATE BEDROOM,50336791874,verified,Tucker,Brooklyn,Kensington,40.64302,-73.97255,United States,US,False,flexible,Private room,2015,779.0,156.0,-1,82,2019-05-19,0.94,2,2,131,"no inside smoking, quiet(ish) after 11, help y...",NaN
421,1233854,Charming Nolita Apartment!!,7389895192,verified,Belinda,Manhattan,Nolita,40.72094,-73.99706,United States,US,False,flexible,Entire home/apt,2008,874.0,175.0,-10,68,2019-06-10,0.69,5,1,13,NaN,NaN
441,1244900,Cozy apartment in a brownstone,81186886194,verified,Adelaide,Manhattan,Harlem,40.80497,-73.95016,United States,US,False,moderate,Entire home/apt,2021,920.0,184.0,-12,203,2019-07-06,2.14,5,3,77,NaN,NaN
478,1265335,Charming upper west side apartment,89878315253,unconfirmed,Alen,Manhattan,Upper West Side,40.77886,-73.98042,United States,US,True,strict,Entire home/apt,2022,410.0,82.0,-2,129,2019-06-07,1.33,2,1,381,Please remember that this is a residential bui...,NaN
525,1291294,Chateau Style Brooklyn Loft for Singles or Cou...,2631536622,verified,Carlos,Brooklyn,Bedford-Stuyvesant,40.68967,-73.95445,United States,US,False,moderate,Entire home/apt,2022,413.0,83.0,-3,42,2019-05-18,0.44,5,1,292,BUILDING IS RESIDENTIAL. DOCTORS AND NURSES WH...,NaN
42446,24444262,"Cozy room in bright, spacious apartment",84040511136,verified,Steven,Bronx,Hunts Point,40.81731,-73.89052,United States,US,False,moderate,Private room,2003,1200.0,240.0,-1223,0,NaT,NaN,2,4,341,Smoking is strictly prohibited. Your pets are ...,NaN
42500,24474086,2bd BOUTIQUE Apartament in the heart of MANHA...,2679070022,unconfirmed,Tom,Manhattan,Hell's Kitchen,40.76694,-73.98773,United States,US,True,flexible,Entire home/apt,2009,711.0,142.0,-365,13,2019-07-07,5.91,4,4,0,Please pick up/drop packet with front desk con...,NaN
42538,24495073,Newly Renovated Garden Apartment,98469733112,verified,Margie,Brooklyn,Bedford-Stuyvesant,40.68470,-73.94350,United States,US,True,moderate,Entire home/apt,2022,85.0,17.0,-200,3,2019-04-23,1.06,2,1,157,No smoking or pets allowed and we request that...,NaN


In [31]:
# Rows where minimum_nights is less than 1 or greater than 365;

outlier_mask = (df["minimum_nights"] < 1) | (df["minimum_nights"] > 365)

# Count and Rate;

outlier_count = outlier_mask.sum()
outlier_ratio = (outlier_count / len(df)) * 100

print(f"Total number of rows less than 1 or greater than 365: {outlier_count}")
print(f"Percentage of total data: {outlier_ratio:.4f}%")

Total number of rows less than 1 or greater than 365: 48
Percentage of total data: 0.0468%


In [32]:
df = df[(df["minimum_nights"] >= 1) & (df["minimum_nights"] <= 365)].copy()

Diagnostic checks revealed 48 rows where minimum_nights fell outside the valid range of [1, 365]. 
Because these anomalous entries represent an extremely negligible portion of the dataset (0.0468%), removing them outright cleanses the feature without introducing bias or reducing the statistical power of subsequent analysis.

In [33]:
print(f"Updated Total Row Count: {len(df)}")
print(f"minimum_nights range: {df['minimum_nights'].min()} - {df['minimum_nights'].max()}")

Updated Total Row Count: 102142
minimum_nights range: 1 - 365


In [34]:
# Count and ratio of availability_365 values exceeding 365 days;

avail_over_365_count = (df["availability_365"] > 365).sum()
avail_over_365_ratio = (avail_over_365_count / len(df)) * 100

print(f"Number of rows with availability > 365: {avail_over_365_count}")
print(f"Percentage of total data: {avail_over_365_ratio:.4f}%")

Number of rows with availability > 365: 2765
Percentage of total data: 2.7070%


In [35]:
# Count of negative values in availability_365;

avail_neg_count = (df["availability_365"] < 0).sum()
print(f"Number of negative availability values: {avail_neg_count}")

Number of negative availability values: 426


Combined, these anomalous entries account for 3.1240% of the dataset. Since dropping these rows would needlessly discard valid information across other attributes (e.g., location, price, room type), converting only these specific cells to NaN eliminates measurement error while preserving overall dataset completeness for Power BI modeling.

In [36]:
# availability_365: Flagging negative (<0) and values exceeding 365 as cell-level NaN;

df.loc[(df["availability_365"] < 0) | (df["availability_365"] > 365), "availability_365"] = np.nan

# Final verification;

print(f"Remaining Total Row Count: {len(df)}")
print("availability_365 'NaN' count:", df["availability_365"].isna().sum())
print("availability_365 min - max:", df["availability_365"].min(), "-", df["availability_365"].max())

Remaining Total Row Count: 102142
availability_365 'NaN' count: 3615
availability_365 min - max: 0 - 365


**Outlier Analysis & Cleaning Strategy Conclusion**

During the outlier analysis, physically impossible values (negative numbers and values exceeding the 365-day annual limit) were identified in the minimum_nights and availability_365 features. The following actions were executed:

**minimum_nights Feature:** A total of 48 rows (0.0468% of the dataset) contained logically invalid values (less than 1 night or greater than 365 nights). Since the impact of this removal on sample representativeness is negligible, these 48 rows were filtered out entirely.

**availability_365 Feature:** A total of 3,191 invalid observations (3.1240% of the dataset) were identified, consisting of negative values (426 rows) and records exceeding 365 days (2,765 rows). Dropping 3% of the dataset would result in significant information loss, stripping away valid, highly valuable attributes such as price, location, and room type. To prevent unnecessary data loss, a cell-level cleaning approach was adopted, converting these specific invalid entries to missing values (NaN).

**Outcome:** This dual-strategy ensures that calculated metrics in Power BI (e.g., average length of stay, annual availability rates) remain uncorrupted while fully preserving the dimensional integrity of listings for downstream analysis.

In [37]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
id,102142.0,29190159.041824,1001254.0,15138697.0,29192661.0,43239445.0,57367417.0,16248338.036273
host_id,102142.0,49258329726.053757,123600518.0,24583294859.0,49121436885.0,74006192315.0,98763129024.0,28537968280.756256
lat,102134.0,40.728089,40.49979,40.688733,40.72229,40.76277,40.91697,0.055866
long,102134.0,-73.949623,-74.24984,-73.98256,-73.95444,-73.93233,-73.70522,0.049526
construction_year,101940.0,2012.486747,2003.0,2007.0,2012.0,2017.0,2022.0,5.764688
price,101895.0,625.322302,50.0,340.0,625.0,913.0,1200.0,331.664484
service_fee,101869.0,125.032463,10.0,68.0,125.0,183.0,240.0,66.324395
minimum_nights,102142.0,7.856513,1.0,2.0,3.0,5.0,365.0,17.05118
number_of_reviews,101961.0,27.43649,0.0,1.0,7.0,30.0,1024.0,49.393521
last_review,86330,2019-06-12 01:39:24.832619008,2012-07-11 00:00:00,2018-10-28 00:00:00,2019-06-14 00:00:00,2019-07-05 00:00:00,2058-06-16 00:00:00,NaN


## <span style=" color:red"> Missing Value Inspection and Handling

In [38]:
df.isnull().sum().sort_values(ascending=False) # Checking Missing Values.

license                           102140
house_rules                        51920
last_review                        15812
reviews_per_month                  15804
availability_365                    3615
country                              531
host_name                            404
calculated_host_listings_count       319
review_rate_number                   305
host_identity_verified               278
service_fee                          273
price                                247
name                                 242
construction_year                    202
number_of_reviews                    181
country_code                         122
instant_bookable                      96
cancellation_policy                   75
neighbourhood_group                   28
neighbourhood                         15
lat                                    8
long                                   8
host_id                                0
id                                     0
minimum_nights  

In [39]:
df.isnull().sum().sum() # Total missing value.

np.int64(192625)

In [40]:
(df.isnull().mean() * 100).sort_values(ascending=False).round(2) # Missing Value Rates.

license                           100.00
house_rules                        50.83
last_review                        15.48
reviews_per_month                  15.47
availability_365                    3.54
country                             0.52
host_name                           0.40
calculated_host_listings_count      0.31
review_rate_number                  0.30
host_identity_verified              0.27
service_fee                         0.27
price                               0.24
name                                0.24
construction_year                   0.20
number_of_reviews                   0.18
country_code                        0.12
instant_bookable                    0.09
cancellation_policy                 0.07
neighbourhood_group                 0.03
neighbourhood                       0.01
lat                                 0.01
long                                0.01
host_id                             0.00
id                                  0.00
minimum_nights  

In [41]:
df[df["license"].notna()]

# The 'license' column is missing in all but two rows.
# Since there is insufficient data to impute missing values or perform meaningful analysis, we drop this column.

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license
11114,7139598,"Cozy 1 BR on Bedford Avenue, Wburg",73023181304,verified,Christina,Brooklyn,Williamsburg,40.71764,-73.95689,United States,US,True,strict,Private room,2010,702.0,140.0,1,1,2016-01-03,0.02,1,1,191,"Dear Guest, Thank you for appreciating that I ...",41662/AL
72947,41289964,"Cozy 1 BR on Bedford Avenue, Wburg",25804773951,unconfirmed,Christina,Brooklyn,Williamsburg,40.71764,-73.95689,United States,US,True,flexible,Private room,2010,702.0,140.0,1,1,2016-01-03,0.02,1,1,0,NaN,41662/AL


In [42]:
df.drop(columns=["license"], inplace=True)

In [43]:
df["house_rules"].head(5)

# Because these values cannot be reliably or accurately imputed without introducing synthetic bias, 
# they have been intentionally left as NaN. 

0    Clean up and treat the home the way you'd like...
1    Pet friendly but please confirm with me if the...
2    I encourage you to use my kitchen, cooking and...
3                                                  NaN
4    Please no smoking in the house, porch or on th...
Name: house_rules, dtype: object

In [44]:
df["country"].unique() 

# Since our dataset is centered on New York Airbnb listings and all existing unique non-null values are 'United States',
# we decided to impute the null values with 'United States'.

array(['United States', nan], dtype=object)

In [45]:
df["country"] = df["country"].fillna("United States")

In [46]:
df["country_code"].unique()

# Similarly, all null values will be imputed with 'US'.

array(['US', nan], dtype=object)

In [47]:
df["country_code"] = df["country_code"].fillna("US")

In [48]:
df[["last_review","reviews_per_month","review_rate_number","number_of_reviews"]].head(10)

# These columns have missing values across all four fields, or at least the first three. 
# Verification shows that the remaining columns in these rows contain complete data, 
# indicating that these listings simply received no reviews. 
# Therefore, we retain these rows as they provide valuable data for other analytical dimensions.

# Additionally, some rows have NaN for 'last_review' and 'reviews_per_month', and 0 for 'number_of_reviews', 
# yet contain a valid 'review_rate_number'. 
# Lacking internal documentation on the rating system architecture, we cannot draw definitive conclusions. 
# However, this discrepancy likely stems from a different evaluation workflow—for instance, 
# listings that received a star rating without a written review.

,last_review,reviews_per_month,review_rate_number,number_of_reviews
0,2021-10-19,0.21,4,9
1,2022-05-21,0.38,4,45
2,NaT,NaN,5,0
3,2019-07-05,4.64,4,270
4,2018-11-19,0.10,3,9
5,2019-06-22,0.59,3,74
6,2017-10-05,0.40,5,49
7,2017-10-05,0.40,5,49
8,2019-06-24,3.47,3,430
9,2017-07-21,0.99,5,118


In [49]:
df[
    df["last_review"].isna() &
    df["reviews_per_month"].isna() &
    df["review_rate_number"].isna() &
    (df["number_of_reviews"] == 0)
]

# When checking rows where 'last_review', 'reviews_per_month', and 'review_rate_number' are null while 'number_of_reviews' is 0,
# we observe that all other columns contain complete data.
# Therefore, we can infer that these listings simply have not received any reviews or ratings yet.
# To avoid data loss, we retain these rows as they provide valuable insights across other features.

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
27,1016248,Magnifique Suite au N de Manhattan - vue Cloitres,38811420224,verified,Adrianna,Manhattan,Inwood,40.86754,-73.92639,United States,US,<NA>,strict,Private room,2017,274.0,55.0,4,0,NaT,NaN,<NA>,1,96,To treat our home with respect. No smoking in...
37,1021771,Clean and Quiet in Brooklyn,26207748876,verified,Arthur,Brooklyn,Bedford-Stuyvesant,40.68876,-73.94312,United States,US,False,moderate,Private room,2004,203.0,41.0,60,0,NaT,NaN,<NA>,1,294,NO Shoes in the house. This is why my house is...
266,1148248,"HOSTING YOUR SUNNY, SPACIOUS NYC ROOM",84001079377,verified,NaN,Manhattan,Inwood,40.86648,-73.92630,United States,US,False,strict,Private room,2003,373.0,75.0,7,0,NaT,NaN,<NA>,2,<NA>,NaN
268,1149352,Prime East Village 1 Bedroom,28535518402,unconfirmed,NaN,Manhattan,East Village,40.72807,-73.98594,United States,US,True,flexible,Entire home/apt,2012,136.0,27.0,3,0,NaT,NaN,<NA>,1,<NA>,1) We live in a quiet neighborhood so no loud ...
487,1270306,Very close to Downtown Awesome Private Apartment,7445813909,unconfirmed,Antony,Brooklyn,Gravesend,40.60452,-73.97103,United States,US,True,flexible,Entire home/apt,2021,762.0,152.0,7,0,NaT,NaN,<NA>,2,159,NaN
659,1365302,1 Bedroom Available In My Two Bedroom Flat,37408540678,verified,Scott,Brooklyn,Crown Heights,40.67495,-73.95563,United States,US,False,flexible,Private room,2009,69.0,14.0,3,0,NaT,NaN,<NA>,1,158,All Renters and Guests must be 25 years of age...
792,1438758,One bedroom sharing Bathroom,26184450237,unconfirmed,Harris,Brooklyn,Cypress Hills,40.67889,-73.86404,United States,US,False,flexible,Private room,2012,904.0,181.0,7,0,NaT,NaN,<NA>,1,255,NaN
794,1439862,Huge Williamsburg Loft..Perfect for Big Groups!,74128844694,verified,Tucker,Brooklyn,Williamsburg,40.70766,-73.95191,United States,US,True,flexible,Entire home/apt,2005,777.0,155.0,7,0,NaT,NaN,<NA>,2,109,It is VERY important to me that you respect my...
860,1476314,Fully Furnished 1B/1BTH UWS GEM 1 YR Sublease,44588815342,verified,Stewart,Manhattan,Upper West Side,40.78012,-73.98439,United States,US,True,moderate,Entire home/apt,2007,316.0,63.0,1,0,NaT,NaN,<NA>,1,302,"Unlike most other Airbnb listings, what we do ..."
12148,7710676,Stunning 2 bd home in historic limestone,38570550324,unconfirmed,Crystal,Brooklyn,Bedford-Stuyvesant,40.69156,-73.94276,United States,US,False,flexible,Entire home/apt,2007,453.0,91.0,7,0,NaT,NaN,<NA>,1,277,We expect travelers to leave the apartment in ...


In [50]:
df[
    df["neighbourhood_group"].isna() |
    df["neighbourhood"].isna()
]

# Buradaki null değerlere baktığımızda; neighbourhood_group'ı boş olanların neighbourhood'ının dolu,
# neighbourhood'ı dolu olanların ise neighbourhood_group'ının boş olduğunu görüyoruz. 
# Bu nedenle, boşlukları sağlıklıca doldurabilecek miyiz diye bakacağız.

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
74,1042206,"HARLEM, NEW YORK WELCOMES YOU!!",98195975718,NaN,Violet,NaN,Washington Heights,40.83139,-73.94095,United States,US,True,moderate,Private room,2011,571.0,114.0,2,49,2019-06-18,1.60,2,2,<NA>,The usual courtesies apply: - No smoking - No ...
75,1042759,BLUE TRIM GUEST HOUSE,4726877402,unconfirmed,Audrey,NaN,Clinton Hill,40.68346,-73.96374,United States,US,True,strict,Private room,2014,398.0,80.0,2,105,2019-06-26,0.92,1,1,<NA>,Shoes off please Cat can go in or out as he de...
76,1043311,Charming East Village One Bedroom Flat,74322993447,verified,Violet,NaN,East Village,40.72828,-73.98801,United States,US,False,strict,Entire home/apt,2018,618.0,124.0,5,21,2019-01-02,0.20,4,1,<NA>,no smoking quiet
77,1043863,Manhattan Room,11468499446,verified,Sofia,NaN,Upper East Side,40.76865,-73.95058,United States,US,False,strict,Private room,2007,116.0,23.0,1,142,2019-07-06,1.50,4,1,<NA>,I'm a semi kosher vegetarian which means that ...
78,1044415,Little King of Queens,68599531533,unconfirmed,Melanie,NaN,Woodside,40.75038,-73.90334,United States,US,True,flexible,Private room,2012,54.0,11.0,30,25,NaT,0.22,2,1,<NA>,No Street Shoes allowed in House. No cooking K...
90,1051043,Cozy Bedroom in Williamsburg 3 BR,14067827221,unconfirmed,Lilianna,NaN,Williamsburg,40.71156,-73.96218,United States,US,False,moderate,Private room,2015,266.0,53.0,3,174,2019-06-22,1.54,5,4,<NA>,Dryer and Washing Machine are in Basement (1.0...
91,1051595,Sunny room+Pvte office in huge loft,12884105458,NaN,Albert,NaN,Bushwick,40.70032,-73.93830,United States,US,False,moderate,Private room,2012,728.0,146.0,4,24,NaT,0.28,5,1,<NA>,"To enjoy, relax, feel safe and cozy. Also, kee..."
92,1052148,Spacious Prospect Heights Apartment,63218812094,unconfirmed,Sarah,NaN,Prospect Heights,40.68233,-73.97261,United States,US,False,flexible,Entire home/apt,2021,583.0,117.0,4,166,2019-06-27,3.40,2,1,<NA>,We ask that guests be respectful as there are ...
148,1083076,NYC Zen,83696952551,verified,Amelia,NaN,East Village,40.72354,-73.98295,United States,US,False,strict,Entire home/apt,2003,NaN,119.0,3,30,2019-06-17,0.28,5,1,344,No Smoking No Pets No Parties
161,1090256,Indie-Chic Share In Williamsburg,1595619477,unconfirmed,Darcy,NaN,Williamsburg,40.71088,-73.95055,United States,US,<NA>,NaN,Private room,2022,1020.0,204.0,4,202,2019-05-28,1.86,5,2,<NA>,"No smoking in the apartment, even with the win..."


In [51]:
# We are performing a null value inspection for the 'neighbourhood' and 'neighbourhood_group' columns.
# This code displays the mapping between each 'neighbourhood' and its corresponding 'neighbourhood_group';

df.groupby("neighbourhood")["neighbourhood_group"].unique()

neighbourhood
Allerton                                         [Bronx]
Arden Heights                            [Staten Island]
Arrochar                                 [Staten Island]
Arverne                                         [Queens]
Astoria                                         [Queens]
Bath Beach                                    [Brooklyn]
Battery Park City                            [Manhattan]
Bay Ridge                                     [Brooklyn]
Bay Terrace                                     [Queens]
Bay Terrace, Staten Island               [Staten Island]
Baychester                                       [Bronx]
Bayside                                         [Queens]
Bayswater                                       [Queens]
Bedford-Stuyvesant                       [Brooklyn, nan]
Belle Harbor                                    [Queens]
Bellerose                                       [Queens]
Belmont                                          [Bronx]
Bensonhurst      

In [52]:
df.groupby("neighbourhood")["neighbourhood_group"].nunique().sort_values(ascending=False)

# To guarantee that missing geographic attributes can be imputed deterministically, 
# we evaluate the mapping cardinality between neighbourhood and neighbourhood_group. 
# The diagnostic shows that every neighbourhood belongs to a single, unique group—with the exception of Chelsea and South Slope, 
# which span multiple boundaries.

neighbourhood
Chelsea                       2
South Slope                   2
Arrochar                      1
Allerton                      1
Astoria                       1
Bath Beach                    1
Battery Park City             1
Arden Heights                 1
Bay Terrace                   1
Bay Terrace, Staten Island    1
Baychester                    1
Bayside                       1
Bayswater                     1
Bedford-Stuyvesant            1
Belle Harbor                  1
Arverne                       1
Belmont                       1
Bensonhurst                   1
Bergen Beach                  1
Boerum Hill                   1
Borough Park                  1
Breezy Point                  1
Briarwood                     1
Brighton Beach                1
Bronxdale                     1
Brooklyn Heights              1
Brownsville                   1
Bull's Head                   1
Bushwick                      1
Cambria Heights               1
Canarsie                  

In [53]:
df[df["neighbourhood"] == "Chelsea"]["neighbourhood_group"].unique()

array(['manhatan', 'Manhattan', nan], dtype=object)

In [54]:
df[df["neighbourhood"] == "South Slope"]["neighbourhood_group"].unique()

# Upon closer inspection, both actually belong to a single group, 
# but they appeared distinct due to typos/formatting inconsistencies.

array(['brookln', 'Brooklyn'], dtype=object)

In [55]:
df["neighbourhood_group"] = df["neighbourhood_group"].replace({
    "brookln": "Brooklyn",
    "manhatan": "Manhattan"
})

# Standardizing string formats to correct typos and spacing inconsistencies.

In [56]:
df[df["neighbourhood_group"].isna()]["neighbourhood"].map(
    df.groupby("neighbourhood")["neighbourhood_group"].first()
)

# Here, for rows with a missing 'neighbourhood_group', we impute the value 
# by referencing the 'neighbourhood_group' associated with the same 'neighbourhood' in other rows.

74       Manhattan
75        Brooklyn
76       Manhattan
77       Manhattan
78          Queens
90        Brooklyn
91        Brooklyn
92        Brooklyn
148      Manhattan
161       Brooklyn
168       Brooklyn
196      Manhattan
206       Brooklyn
241      Manhattan
249       Brooklyn
262          Bronx
295      Manhattan
307      Manhattan
319       Brooklyn
361         Queens
384      Manhattan
415       Brooklyn
448       Brooklyn
465      Manhattan
488      Manhattan
492      Manhattan
40383     Brooklyn
40396    Manhattan
Name: neighbourhood, dtype: object

In [57]:
df.groupby("neighbourhood")["neighbourhood_group"].unique()

neighbourhood
Allerton                               [Bronx]
Arden Heights                  [Staten Island]
Arrochar                       [Staten Island]
Arverne                               [Queens]
Astoria                               [Queens]
Bath Beach                          [Brooklyn]
Battery Park City                  [Manhattan]
Bay Ridge                           [Brooklyn]
Bay Terrace                           [Queens]
Bay Terrace, Staten Island     [Staten Island]
Baychester                             [Bronx]
Bayside                               [Queens]
Bayswater                             [Queens]
Bedford-Stuyvesant             [Brooklyn, nan]
Belle Harbor                          [Queens]
Bellerose                             [Queens]
Belmont                                [Bronx]
Bensonhurst                         [Brooklyn]
Bergen Beach                        [Brooklyn]
Boerum Hill                         [Brooklyn]
Borough Park                        [Brooklyn]

In [58]:
df["neighbourhood_group"] = df["neighbourhood_group"].fillna(
    df.groupby("neighbourhood")["neighbourhood_group"].transform("first")
)

# Here we impute missing values:
# As established previously, every 'neighbourhood' maps to a specific 'neighbourhood_group' (though some contain nulls).
# This code retrieves the first non-null group value for each neighborhood name
# and uses it to fill the corresponding NaN entries in 'neighbourhood_group'.

In [59]:
df["neighbourhood_group"].isna().sum()

np.int64(0)

In [60]:
df.isnull().sum()

id                                    0
name                                242
host_id                               0
host_identity_verified              278
host_name                           404
neighbourhood_group                   0
neighbourhood                        15
lat                                   8
long                                  8
country                               0
country_code                          0
instant_bookable                     96
cancellation_policy                  75
room_type                             0
construction_year                   202
price                               247
service_fee                         273
minimum_nights                        0
number_of_reviews                   181
last_review                       15812
reviews_per_month                 15804
review_rate_number                  305
calculated_host_listings_count      319
availability_365                   3615
house_rules                       51920


In [61]:
# Next, we impute the missing values in the 'neighbourhood' column;

df[df["neighbourhood"].isna()][
    ["neighbourhood", "neighbourhood_group", "lat", "long"]
]

# For the rows with missing 'neighbourhood' values, we examined the remaining features to see if any inferences could be made.
# As it turns out, all other columns for these rows are completely filled.

,neighbourhood,neighbourhood_group,lat,long
517,NaN,Brooklyn,40.71580,-73.95803
547,NaN,Manhattan,40.73089,-73.98195
575,NaN,Manhattan,40.79816,-73.96190
589,NaN,Brooklyn,40.68012,-73.97847
613,NaN,Manhattan,40.72709,-73.98274
624,NaN,Manhattan,40.75348,-73.97065
633,NaN,Manhattan,40.71693,-73.98948
643,NaN,Brooklyn,40.68016,-73.94878
670,NaN,Brooklyn,40.73641,-73.95330
678,NaN,Brooklyn,40.73693,-73.95316


In [62]:
from sklearn.metrics import pairwise_distances

missing = df[df["neighbourhood"].isna()]

known = df[df["neighbourhood"].notna()]

for idx, row in missing.iterrows():
    distances = (
        (known["lat"] - row["lat"])**2 +
        (known["long"] - row["long"])**2
    )
    
    nearest = known.loc[distances.nsmallest(5).index,
                        ["neighbourhood", "neighbourhood_group", "lat", "long"]]
    
    print(f"\nEksik index: {idx}")
    print(nearest)

# To resolve the 16 missing neighbourhood entries, 
# we leverage spatial proximity by comparing their exact geographic coordinates (lat/long) against established neighborhood centroids. 
# With the exception of 2 boundary cases that yielded multiple close candidates, 
# all remaining locations were deterministically mapped to a single nearest neighborhood.


Eksik index: 517
      neighbourhood neighbourhood_group       lat      long
2565   Williamsburg            Brooklyn  40.71571 -73.95813
70741  Williamsburg            Brooklyn  40.71571 -73.95813
27148  Williamsburg            Brooklyn  40.71563 -73.95815
27260  Williamsburg            Brooklyn  40.71564 -73.95821
35803  Williamsburg            Brooklyn  40.71567 -73.95831

Eksik index: 547
      neighbourhood neighbourhood_group       lat      long
8669   East Village           Manhattan  40.73082 -73.98182
26115  East Village           Manhattan  40.73105 -73.98195
68634  East Village           Manhattan  40.73105 -73.98195
83279  East Village           Manhattan  40.73105 -73.98195
7466   East Village           Manhattan  40.73097 -73.98178

Eksik index: 575
         neighbourhood neighbourhood_group       lat      long
58018  Upper West Side           Manhattan  40.79830 -73.96204
60410  Upper West Side           Manhattan  40.79809 -73.96209
51343  Upper West Side           Manh

In [63]:
idx = 633
row = df.loc[idx]

known = df[df["neighbourhood"].notna()]

distances = (
    (known["lat"] - row["lat"])**2 +
    (known["long"] - row["long"])**2
)

known.loc[
    distances.nsmallest(15).index,
    ["neighbourhood", "neighbourhood_group", "lat", "long"]
]

# We wanted to inspect row index 633 in detail because it yielded 2 different candidate neighbourhoods.
# By analyzing its nearest 15 neighboring records, we obtained 2 distinct potential results.

,neighbourhood,neighbourhood_group,lat,long
37566,Lower East Side,Manhattan,40.71707,-73.98931
3468,Lower East Side,Manhattan,40.71715,-73.98936
12441,Chinatown,Manhattan,40.71668,-73.98954
14594,Chinatown,Manhattan,40.71659,-73.98949
84416,Chinatown,Manhattan,40.71659,-73.98949
307,Chinatown,Manhattan,40.71659,-73.98945
8687,Lower East Side,Manhattan,40.71661,-73.98916
14615,Lower East Side,Manhattan,40.71713,-73.98906
84437,Lower East Side,Manhattan,40.71713,-73.98906
14773,Chinatown,Manhattan,40.71674,-73.98995


In [64]:
missing = df[df["neighbourhood"].isna()]

for idx, row in missing.iterrows():
    same_location = df[
        (df["lat"] == row["lat"]) &
        (df["long"] == row["long"]) &
        (df["neighbourhood"].notna())
    ][["neighbourhood", "neighbourhood_group", "lat", "long"]]

    print(f"\nEksik index: {idx}")
    print(same_location)

# Here, we checked whether there were any exact coordinate matches for our 16 records rather than just proximity matches, 
# and found none.


Eksik index: 517
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 547
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 575
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 589
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 613
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 624
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 633
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 643
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 670
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]
Index: []

Eksik index: 678
Empty DataFrame
Columns: [neighbourhood, neighbourhood_group, lat, long]


In [65]:
# Conclusion:

# We imputed the missing values in the 'neighbourhood_group' column because they could be verified 
# using the 'neighbourhood_group' information from other records belonging to the same 'neighbourhood'.

# However, for the 16 missing values in the 'neighbourhood' column, no exact matching coordinates were found. 
# We could only infer potential values based on proximity to nearest coordinates.

# To maintain data integrity and avoid introducing unverified assumptions, we chose not to impute 
# these 16 records and retained them as missing (NaN).

In [66]:
df.isnull().sum().sort_values(ascending=False)

house_rules                       51920
last_review                       15812
reviews_per_month                 15804
availability_365                   3615
host_name                           404
calculated_host_listings_count      319
review_rate_number                  305
host_identity_verified              278
service_fee                         273
price                               247
name                                242
construction_year                   202
number_of_reviews                   181
instant_bookable                     96
cancellation_policy                  75
neighbourhood                        15
long                                  8
lat                                   8
neighbourhood_group                   0
id                                    0
host_id                               0
room_type                             0
country                               0
country_code                          0
minimum_nights                        0


In [67]:
df[df["lat"].isna() | df["long"].isna()]

# We will not impute these values since there are only 8 missing entries. 

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
779,1431578,"Large, furnished room in a 2 bedroom!",20368956893,unconfirmed,Gibson,Brooklyn,Crown Heights,NaN,NaN,United States,US,False,strict,Private room,<NA>,539.0,108.0,1,1,2017-03-18,0.04,2,1,41,- Weekly and monthly prices are much lower - P...
785,1434892,Authentic NY Charming Artist Loft,66486085219,unconfirmed,Bailey,Brooklyn,Greenpoint,NaN,NaN,United States,US,False,strict,Entire home/apt,2021,1058.0,212.0,5,14,2019-06-19,0.16,5,1,226,We live and let live - hoping that you'd be re...
799,1442624,Huge room with private balcony,69386945815,verified,Hunt,Manhattan,East Village,NaN,NaN,United States,US,False,flexible,Private room,2010,506.0,101.0,6,1,2013-05-06,0.01,1,1,240,Expect respect for the family and the space--t...
814,1450908,Decorators 5-Star Flat West Village,33280739304,verified,Watson,Manhattan,West Village,NaN,NaN,United States,US,True,strict,Entire home/apt,2003,381.0,76.0,20,157,2016-08-11,1.71,4,1,61,"Please keep it clean, thats all we really ask ..."
843,1466925,Nice Private Room Beauty in Queens,15305733205,verified,Roberts,Queens,Elmhurst,NaN,NaN,United States,US,True,strict,Private room,2005,224.0,45.0,1,63,2019-05-18,0.89,3,2,70,NaN
885,1490122,Cute Room in Historic Loft!,42267829819,unconfirmed,Jones,Brooklyn,Greenpoint,NaN,NaN,United States,US,True,flexible,Private room,2019,524.0,105.0,14,22,2019-05-02,0.25,1,1,266,"Pets are cool (just clean up after them!), smo..."
926,1512766,21 day Chelsea Apartment rental,10876728736,unconfirmed,Owens,Manhattan,Flatiron District,NaN,NaN,United States,US,False,strict,Private room,2020,623.0,125.0,21,0,NaT,NaN,2,1,104,NaN
986,1545904,New York City for All Seasons!,26437872336,unconfirmed,Douglas,Manhattan,Upper West Side,NaN,NaN,United States,US,True,flexible,Private room,2014,413.0,83.0,1,25,2013-06-22,0.28,2,1,259,No Smoking No Pets


In [68]:
df[df["cancellation_policy"].isna()]

# No transformation or imputation was performed.

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
156,1087495,"Sunny, clean 1 bdrm in W. Village",75987317883,verified,James,Manhattan,West Village,40.73226,-74.00401,United States,US,<NA>,NaN,Entire home/apt,2018,738.0,148.0,45,134,NaT,1.24,3,1,<NA>,NaN
157,1088047,Great location in Williamsburg,51756716507,unconfirmed,Roman,Brooklyn,Williamsburg,40.71363,-73.96398,United States,US,<NA>,NaN,Entire home/apt,2007,1013.0,203.0,6,27,NaT,0.25,3,1,<NA>,NaN
158,1088599,Light and Airy Upper East Side 1 BDR apartment,30351473270,unconfirmed,Kellan,Manhattan,Upper East Side,40.77711,-73.95270,United States,US,<NA>,NaN,Entire home/apt,2004,464.0,93.0,4,126,NaT,1.16,1,2,280,NaN
159,1089152,Luxury Brownstone in Boerum Hill,22525653405,unconfirmed,Owen,Brooklyn,Boerum Hill,40.68559,-73.98094,United States,US,<NA>,NaN,Entire home/apt,2006,744.0,149.0,3,23,NaT,0.27,2,1,<NA>,There is no smoking in or immediately around t...
160,1089704,CENTRAL PARK LOFT all for YOU,82585795978,verified,Eddy,Manhattan,Upper East Side,40.77456,-73.95323,United States,US,<NA>,NaN,Entire home/apt,2014,764.0,153.0,1,234,2019-06-08,2.60,5,2,353,NaN
161,1090256,Indie-Chic Share In Williamsburg,1595619477,unconfirmed,Darcy,Brooklyn,Williamsburg,40.71088,-73.95055,United States,US,<NA>,NaN,Private room,2022,1020.0,204.0,4,202,2019-05-28,1.86,5,2,<NA>,"No smoking in the apartment, even with the win..."
162,1090809,"A room w/ a Manhattan view, longer stay",973585651,unconfirmed,Dominik,Queens,Sunnyside,40.74559,-73.92313,United States,US,<NA>,NaN,Private room,2018,355.0,71.0,30,28,2019-04-12,0.26,1,1,<NA>,- We have a very friendly cat and two chickens...
163,1091361,"Private, Large & Sunny 1BR w/W&D",86944769515,verified,Aldus,Brooklyn,Bedford-Stuyvesant,40.68306,-73.94659,United States,US,<NA>,NaN,Entire home/apt,2008,244.0,49.0,2,309,2019-06-22,NaN,4,2,169,"Please keep cats out of bedrooms and bathroom,..."
199,1111244,Big Room/Washer-Dryer/Wifi/AC/JMZ,43945171386,verified,NaN,Brooklyn,Bedford-Stuyvesant,40.69546,-73.93503,United States,US,<NA>,NaN,Private room,<NA>,457.0,91.0,2,11,2017-11-13,0.48,3,1,<NA>,"We try hard to make our home lovely, and ask y..."
200,1111796,cozy studio with parking spot,16211727428,unconfirmed,Dale,Queens,Middle Village,40.71722,-73.87856,United States,US,<NA>,NaN,Entire home/apt,<NA>,938.0,188.0,30,33,2015-05-09,0.31,4,5,<NA>,"Please clean after yourself, wipe spilled wate..."


In [69]:
df[df["service_fee"].isna()][
    ["price", "service_fee", "room_type", "neighbourhood_group", "neighbourhood"]
]

,price,service_fee,room_type,neighbourhood_group,neighbourhood
15,578.0,NaN,Entire home/apt,Manhattan,West Village
16,778.0,NaN,Entire home/apt,Brooklyn,Williamsburg
17,656.0,NaN,Entire home/apt,Brooklyn,Fort Greene
18,460.0,NaN,Private room,Manhattan,Chelsea
19,1095.0,NaN,Entire home/apt,Brooklyn,Crown Heights
2168,1093.0,NaN,Private room,Brooklyn,Bushwick
2169,90.0,NaN,Shared room,Manhattan,East Harlem
2170,1051.0,NaN,Entire home/apt,Manhattan,Upper East Side
2171,891.0,NaN,Private room,Manhattan,Upper West Side
2172,802.0,NaN,Entire home/apt,Brooklyn,South Slope


In [70]:
df[df["price"].isna()][
    ["price", "service_fee", "room_type", "neighbourhood_group", "neighbourhood"]
]

,price,service_fee,room_type,neighbourhood_group,neighbourhood
147,NaN,64.0,Entire home/apt,Brooklyn,Williamsburg
148,NaN,119.0,Entire home/apt,Manhattan,East Village
210,NaN,176.0,Entire home/apt,Brooklyn,Williamsburg
211,NaN,152.0,Private room,Brooklyn,Bedford-Stuyvesant
212,NaN,151.0,Private room,Manhattan,NoHo
213,NaN,48.0,Entire home/apt,Manhattan,West Village
403,NaN,29.0,Entire home/apt,Brooklyn,Greenpoint
404,NaN,56.0,Entire home/apt,Queens,Forest Hills
405,NaN,157.0,Entire home/apt,Manhattan,West Village
516,NaN,83.0,Entire home/apt,Manhattan,Chelsea


In [71]:
# No transformations or imputations were applied to 'price' and 'service_fee'.

**Missing Value Cleaning & Validation Methodology:**

Missing values (NaN) were systematically addressed using domain knowledge, structural consistency checks, and spatial proximity analysis (k-NN approach) rather than arbitrary deletion:

**Metadata Removal:** The license column (100% null) was removed from the dataset.

**Textual Validation & Imputation:** Missing country values were populated based on NYC location context. For rows missing neighbourhood_group, typographical errors (e.g., manhatan, brookln) were corrected before deterministically imputing borough groups via internal neighborhood lookup with 100% accuracy.

**Spatial Proximity Analysis:** Missing sub-neighborhoods were evaluated using Euclidean distance calculations over lat and long coordinates. To eliminate synthetic bias and maintain strict data integrity, 15 records lacking exact coordinate matches were retained in their original state.

**Review & Engagement Features:** Missing values in last_review and reviews_per_month were verified as active listings with zero reviews; these records were retained as NaN to preserve valid core attributes for downstream analytics.

## <span style=" color:red"> Duplicate Value Check and Handling Operations

In [72]:
df.duplicated().sum()

np.int64(532)

In [73]:
duplicates = df[df.duplicated(keep=False)].sort_values(by=list(df.columns)) # Total duplicates.

duplicates

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
9098,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.762220,-73.960300,United States,US,False,moderate,Entire home/apt,2008,105.0,21.0,30,2,2019-06-08,0.21,3,34,157,NaN
102474,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.762220,-73.960300,United States,US,False,moderate,Entire home/apt,2008,105.0,21.0,30,2,2019-06-08,0.21,3,34,157,NaN
9099,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.762490,-73.962170,United States,US,False,moderate,Entire home/apt,2008,285.0,57.0,30,6,2019-01-31,0.14,3,34,67,"The quieter the better, but otherwise make you..."
102475,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.762490,-73.962170,United States,US,False,moderate,Entire home/apt,2008,285.0,57.0,30,6,2019-01-31,0.14,3,34,67,"The quieter the better, but otherwise make you..."
9100,6027266,ACADIA Spacious 2 Bedroom Apt - Close to Hospi...,95854111798,verified,Juliana,Manhattan,Upper East Side,40.760210,-73.961570,United States,US,False,moderate,Entire home/apt,2014,586.0,117.0,30,10,2018-11-18,0.22,5,34,211,No Smoking No Pets
102476,6027266,ACADIA Spacious 2 Bedroom Apt - Close to Hospi...,95854111798,verified,Juliana,Manhattan,Upper East Side,40.760210,-73.961570,United States,US,False,moderate,Entire home/apt,2014,586.0,117.0,30,10,2018-11-18,0.22,5,34,211,No Smoking No Pets
9101,6027818,*ENCHANTMENT* Upper East Side 2 bedroom- Sunny!,73401481508,unconfirmed,Juliana,Manhattan,Upper East Side,40.762440,-73.960310,United States,US,True,moderate,Entire home/apt,2006,539.0,108.0,30,9,2018-09-30,0.20,5,34,<NA>,Please treat it as it were your own home and b...
102477,6027818,*ENCHANTMENT* Upper East Side 2 bedroom- Sunny!,73401481508,unconfirmed,Juliana,Manhattan,Upper East Side,40.762440,-73.960310,United States,US,True,moderate,Entire home/apt,2006,539.0,108.0,30,9,2018-09-30,0.20,5,34,<NA>,Please treat it as it were your own home and b...
9102,6028371,*JAMES* Amazing Spacious 2 Bedroom- Bright!,37678424985,verified,Juliana,Manhattan,Upper East Side,40.760350,-73.961330,United States,US,False,flexible,Entire home/apt,2021,806.0,161.0,30,8,2019-06-11,0.27,4,34,<NA>,Be courteous and respectful to people in the h...
102478,6028371,*JAMES* Amazing Spacious 2 Bedroom- Bright!,37678424985,verified,Juliana,Manhattan,Upper East Side,40.760350,-73.961330,United States,US,False,flexible,Entire home/apt,2021,806.0,161.0,30,8,2019-06-11,0.27,4,34,<NA>,Be courteous and respectful to people in the h...


In [74]:
df.duplicated(keep=False).sum 

<bound method Series.sum of 0         False
1         False
2         False
3         False
4         False
5         False
6         False
7         False
8         False
9         False
10        False
11        False
12        False
13        False
14        False
15        False
16        False
17        False
18        False
19        False
20        False
21        False
22        False
23        False
24        False
25        False
26        False
27        False
28        False
29        False
30        False
31        False
32        False
33        False
34        False
35        False
36        False
37        False
38        False
39        False
40        False
41        False
42        False
43        False
44        False
45        False
56        False
57        False
58        False
59        False
60        False
61        False
62        False
63        False
64        False
65        False
66        False
67        False
68        False
69        False
70        Fa

In [75]:
df = df.drop_duplicates() # Dropping duplicate records.

In [76]:
df.duplicated().sum()

np.int64(0)

In [77]:
df.shape

(101610, 25)

In [78]:
df.sample(10)

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
13501,8457938,Futon in Brooklyn near subway,500740808,unconfirmed,Katherine,Brooklyn,Crown Heights,40.67114,-73.93561,United States,US,False,strict,Shared room,2011,NaN,177.0,1,0,NaT,NaN,4,1,198,NaN
76893,43469340,Clean + Private Bedroom in NYC's Lower East Side,60953618175,verified,Christopher,Manhattan,East Village,40.72025,-73.97901,United States,US,True,strict,Private room,2021,1161.0,232.0,10,14,2018-07-04,0.49,4,1,7,NaN
10757,6942427,Duplex Upper East side,54062126401,verified,Robin,Manhattan,Upper East Side,40.77001,-73.95691,United States,US,True,flexible,Private room,2019,882.0,176.0,2,3,2016-07-18,0.08,3,1,80,"My ideal guests would be warm, friendly, and r..."
10169,6617675,**Dominique's NYC couch bed **sleep*shower & ...,95756435608,unconfirmed,Vie,Bronx,Eastchester,40.88211,-73.83625,United States,US,False,strict,Shared room,2022,760.0,152.0,1,1,2019-04-20,0.37,4,13,350,No smoking (indoors or outside on the property...
77789,43964201,"Не дорогая комната в Нью-Йорке, в Бруклине",50599909803,verified,Ella,Brooklyn,Coney Island,40.57751,-73.98521,United States,US,False,strict,Shared room,2022,662.0,132.0,2,1,2016-12-18,0.03,5,1,0,"Please be aware, check in is NO earlier than 6..."
49388,28278328,Comfortable clean Bedstuy private room,1254085713,verified,Angela,Brooklyn,Bedford-Stuyvesant,40.69551,-73.93951,United States,US,False,moderate,Private room,2010,709.0,142.0,1,2,2019-07-08,2.00,2,1,14,NaN
30127,17640478,"8 mins to Central Park,3 mins to Subway,Guest ...",84205734212,verified,Andrea,Manhattan,Upper East Side,40.77603,-73.95573,United States,US,False,strict,Private room,2018,94.0,19.0,3,35,2019-07-07,2.22,5,1,162,NO Smoking.
94534,53212464,Charming Art Deco Apartment on Central Park,41974629930,unconfirmed,Tony,Manhattan,Upper West Side,40.78925,-73.96757,United States,US,False,moderate,Entire home/apt,2019,568.0,114.0,6,21,2019-06-30,1.83,3,1,116,NaN
51235,29298427,ENTIRE 1 Bedroom Garden Apt in Victorian Home,89331498340,verified,Menachem,Brooklyn,Flatbush,40.63159,-73.96457,United States,US,False,flexible,Private room,2014,406.0,81.0,2,9,2022-02-27,5.40,4,1,350,NaN
101748,57196756,★ NEW 2 BEDROOM APT NEXT TO CENTRAL PARK WEST★,79951939358,unconfirmed,Inna,Manhattan,Upper West Side,40.79584,-73.96242,United States,US,False,moderate,Entire home/apt,2006,400.0,80.0,30,5,2019-05-11,0.21,5,16,200,-Please no pets -Quiet hours are from 10pm - 8...


## <span style=" color:red"> General Overview and Inspection of Column Values

In [79]:
df["room_type"].value_counts(dropna=False)

# No non-sensical values were observed across the columns.

room_type
Entire home/apt    53180
Private room       46114
Shared room         2203
Hotel room           113
Name: count, dtype: int64

In [80]:
df["cancellation_policy"].value_counts(dropna=False)

# No non-sensical values were observed across the columns.

cancellation_policy
moderate    34009
strict      33783
flexible    33743
NaN            75
Name: count, dtype: int64

In [81]:
df["host_identity_verified"].value_counts(dropna=False)

# No non-sensical values were observed across the columns.

host_identity_verified
unconfirmed    50727
verified       50605
NaN              278
Name: count, dtype: int64

In [82]:
df["country"].value_counts(dropna=False)

# No non-sensical values were observed across the columns.

country
United States    101610
Name: count, dtype: int64

In [83]:
df["country_code"].value_counts(dropna=False)

# No non-sensical values were observed across the columns.

country_code
US    101610
Name: count, dtype: int64

In [84]:
df["instant_bookable"].value_counts(dropna=False)

# No non-sensical values were observed across the columns.

instant_bookable
False    50967
True     50547
<NA>        96
Name: count, dtype: Int64

In [85]:
# Check if there are any empty string values across string columns (excluding NaN);

df.select_dtypes("object").apply(
    lambda x: (x.str.strip() == "").sum()
)

name                      0
host_identity_verified    0
host_name                 0
neighbourhood_group       0
neighbourhood             0
country                   0
country_code              0
cancellation_policy       0
room_type                 0
house_rules               0
dtype: int64

In [86]:
df.dtypes

id                                         int64
name                                      object
host_id                                    int64
host_identity_verified                    object
host_name                                 object
neighbourhood_group                       object
neighbourhood                             object
lat                                      float64
long                                     float64
country                                   object
country_code                              object
instant_bookable                         boolean
cancellation_policy                       object
room_type                                 object
construction_year                          Int64
price                                    float64
service_fee                              float64
minimum_nights                             Int64
number_of_reviews                          Int64
last_review                       datetime64[ns]
reviews_per_month   

In [87]:
df.head(2)

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,US,False,strict,Private room,2020,966.0,193.0,10,9,2021-10-19,0.21,4,6,286,Clean up and treat the home the way you'd like...
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,US,False,moderate,Entire home/apt,2007,142.0,28.0,30,45,2022-05-21,0.38,4,2,228,Pet friendly but please confirm with me if the...


### Save Cleaned CSV File (and parquet) & Verify

In [88]:
# Save cleaned CSV file and reload for inspection;

df.to_csv("airbnb_cleaned_yeni.csv", index=False)

In [89]:
# Reload and check the saved dataset;

df1 = pd.read_csv("airbnb_cleaned_yeni.csv")

In [90]:
df1.shape

(101610, 25)

In [91]:
df1.head(2)

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,US,False,strict,Private room,2020.0,966.0,193.0,10,9.0,2021-10-19,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,US,False,moderate,Entire home/apt,2007.0,142.0,28.0,30,45.0,2022-05-21,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...


In [92]:
df1.dtypes

id                                  int64
name                               object
host_id                             int64
host_identity_verified             object
host_name                          object
neighbourhood_group                object
neighbourhood                      object
lat                               float64
long                              float64
country                            object
country_code                       object
instant_bookable                   object
cancellation_policy                object
room_type                          object
construction_year                 float64
price                             float64
service_fee                       float64
minimum_nights                      int64
number_of_reviews                 float64
last_review                        object
reviews_per_month                 float64
review_rate_number                float64
calculated_host_listings_count    float64
availability_365                  

In [93]:
df1.duplicated().sum()

np.int64(0)

In [94]:
df.shape

(101610, 25)

In [95]:
df1.shape

(101610, 25)

In [96]:
# Exporting cleaned dataset to Parquet format for optimal storage and analytical performance;

df1.to_parquet('veri.parquet', index=False)

In [97]:
# Read and open parquet format;

df2 = pd.read_parquet('veri.parquet')

In [98]:
df2.head(2)

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,US,False,strict,Private room,2020.0,966.0,193.0,10,9.0,2021-10-19,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,US,False,moderate,Entire home/apt,2007.0,142.0,28.0,30,45.0,2022-05-21,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...


In [99]:
df2.shape

(101610, 25)

In [100]:
# Total number of rows in the cleaned Parquet file
print("Number of rows in the Parquet file:", len(df2))

# Number of unique IDs
print("Number of unique IDs:", df2["id"].nunique())

Number of rows in the Parquet file: 101610
Number of unique IDs: 101610


# <span style=" color:red">Conclusion & Data Readiness

The data cleaning, validation, and engineering pipeline for the New York City Airbnb dataset has been finalized following a strict data-integrity-first methodology. Rather than relying on arbitrary row deletions or naive statistical imputations, every transformation was executed based on structural verification, domain logic, and spatial analysis.

**Data Type Optimization & Schema Standardization:** Explicitly re-cast key analytical features to their proper schemas (converting dates to datetime64, ratings/counts to integers, and categorical attributes to string types) to ensure strict schema enforcement upon Power BI ingestion.

**Outlier Analysis & Cell-Level Cleaning Strategy:** 

  **minimum_nights Feature:** Identified 48 rows ($0.0468\%$) containing physically impossible values ($< 1$ or $> 365$ nights). Due to its negligible volume, these 48 rows were filtered out entirely.
  **availability_365 Feature:** Identified 3,191 invalid records ($3.1240\%$), consisting of negative values (426 rows) and entries exceeding 365 days (2,765 rows). To prevent significant information loss, a cell-level cleaning approach was adopted: invalid values were converted to NaN, fully preserving the listing's remaining valid dimensions (price, location, room type).
 
**Missing Value Cleaning & Validation Methodology:** 

   **Metadata Removal:** Dropped the license column entirely as it contained 100% missing values.
 
   **Deterministic Borough Repair:** Corrected typographical errors in neighbourhood_group (e.g., manhatan, brookln) and deterministically imputed missing borough groups using internal neighborhood mapping with 100% accuracy.
 
   **Spatial Proximity Analysis (k-NN):** Evaluated missing sub-neighborhoods using Euclidean distance calculations over latitude and longitude. For 15 records lacking exact coordinate matches, values were intentionally retained as NaN to avoid introducing synthetic bias.
 
   **Review & Engagement Logic:** Verified that listings with NaN in last_review and reviews_per_month alongside number_of_reviews == 0 simply represented unreviewed listings. Listings with a valid review_rate_number but missing text reviews were preserved as legitimate evaluation workflows (e.g., star-only ratings). All such rows were retained to preserve dimensional data.
 
**Deduplication & Data Health:** Executed full primary-key and row-level deduplication, eliminating redundant records and verifying zero hidden whitespace or empty string anomalies across string columns.
 
**High-Performance BI Integration:** Exported the sanitized dataset into Parquet format alongside CSV. This preserves exact schema data types, provides high columnar compression, and ensures maximum I/O performance during Power BI reporting.The dataset is now completely sanitized, structurally sound, and fully optimized for DAX modeling, interactive dashboard design, and spatial reporting in Power BI.